In [5]:
import os
os.getcwd()

from pathlib import Path
from stgae.config.load_config import load_config
from stgae.data.preproccesing import get_columns
import pandas as pd

paths = load_config()['paths']    
data_path = Path(paths['raw_data'])

df = pd.read_csv(data_path / 'data.txt', names=get_columns(), sep=' ')

Now I count the average number of sensors per epoch

In [6]:
df.groupby('epoch').size().mean()

np.float64(35.303985595703125)

We have approximately 34 measurements per epoch. Given that the original data comes from 54 sensors... I study if there are particular sensors that have very little data.

In [28]:
number_of_epochs = df['epoch'].nunique()
(df.groupby('moteid').size().sort_values() / number_of_epochs).iloc[:15]

moteid
65407.0    0.000015
6485.0     0.000015
33117.0    0.000015
57.0       0.000046
5.0        0.000534
15.0       0.035660
56.0       0.038254
55.0       0.043503
58.0       0.068695
50.0       0.240189
8.0        0.256012
12.0       0.352142
53.0       0.429245
20.0       0.439987
13.0       0.480728
dtype: float64

I observed that there are some possibly mislabled sensors (65407, 6485, 33117, 56, 58, 57) and some sensors with very little data (5, 18, 50, 8, 12). I discarded them entirely.

In [ ]:
discard = [65407, 6485, 33117, 56, 58, 57, 5, 18, 50, 8, 12]
df = df[(df['moteid'] not in discard)]
df['moteid'].unique()


Drop the rows where there are missing values. We are interested in predicting full observations, that is in every epoch, all feature for the nodes that are not measured on that epoch, but not specific missing values for current rows. 

In [ ]:

df.isna().sum() / len(df)

In [ ]:
# df = df.dropna()

Now we will focus on the day where there is less missing data. We can get this day by counting the number of observations per day.

In [ ]:
# #count rows per day and get max
# day = df.groupby('date').size().idxmax()

# #filter only that day
# df = df[df['date'] == day]
# df.head()

I discarded sensors 12, 30, 14 and 53, given that they have data for less than half of the epochs. To make this decision I also looked at the actual distribution of the sensors, to check that this sensors were not a cluster and all together.

In [ ]:
df = df[(df['moteid'] != 12) & (df['moteid'] != 30) & (df['moteid'] != 14) & (df['moteid'] != 53)]
df['moteid'].unique()

I observed that there isn't data corresponding to sensors 5, 15 and 28 either.  I will exclude those sensors as well.

Finally, I reindex the epochs to range from 0 to number of epochs - 1 and sensors from 0 to number of sensors - 1. This will be useful for later defining the tensors.

In [ ]:
first_epoch = df['epoch'].min()
df['epoch'] = df['epoch'] - first_epoch

assert df['epoch'].max() == df['epoch'].nunique()-1

In [ ]:
#rename moteid
moteid_mapping = {old_id: new_id for new_id, old_id in enumerate(sorted(df['moteid'].unique()))}
df['moteid'] = df['moteid'].map(moteid_mapping)
df['moteid'].unique()

Let's build the graph.
I decided to use a graph using knn (with k=5), and the edges weighted according to the inverse of the distance. 

Building adjacency matrix

In [ ]:
from stgae.data.preproccesing import calculate_distances, calculate_adjacency_matrix

coordinates = pd.read_csv(paths['data_root'] / 'sensor_coordinates.txt', sep=' ')

dist_matrix = calculate_distances(coordinates) #numpy array

k=3 #for k-nearest neighbors adj matrix
exclude_sensors = [5, 12, 14, 15, 28, 30, 53]

A = calculate_adjacency_matrix(dist_matrix, k=k, exclude=exclude_sensors) #torch tensor

assert A.shape[0] == len(df['moteid'].unique())

Now, I built the tensors to then create the Dataset.
I created X and M, of shapes
X: shape (T, N, F)
M: shape (T, N, 1)
where
X[t, n] = features of sensor n at epoch t
M[t, n] = 1 if sensor exists at epoch t, else 0

In [ ]:
from stgae.data.preproccesing import build_tensors

#shapes:
#X: (time_steps, num_sensors, num_features)
#M: (time_steps, num_sensors)
X, M = build_tensors(df, epochs=df['epoch'].unique(), sensors=df['moteid'].unique(), feature_cols=['temperature', 'humidity', 'light', 'voltage'])

Now lets build the dataset.  <br>
For a center time t, and a window size W: <br>
past window: [t-W, ..., t] <br>
future window: [t, ..., t+W] <br>

Masking: masking sensor level per window, that is, for each timestep, choose a subset of known sensors at that the center timestep and mask them in that timestep

In [ ]:
X[0].shape, M[0].shape

In [ ]:
from stgae.data.dataset import STBGNNDataset
from stgae.data.preproccesing import StandardScaler

train_split = 0.8
T = X.shape[0]
train_size = int(T * train_split)

#divide into train and test, keeping temporal order, to avoid leakage and learn temporal dependencies
X_train = X[:train_size]
X_test = X[train_size:]

#i fit the scalar on the training data only to prevent leakeage
scaler = StandardScaler()
scaler.fit(X_train)

train_norm = scaler.transform(X_train)
test_norm = scaler.transform(X_test)

train_dataset = STBGNNDataset(train_norm, M, window_size=4, mask_ratio=0.5, split="train")
test_dataset = STBGNNDataset(test_norm, M, window_size=4, mask_ratio=0.5, split="test")

for k in train_dataset[0].keys():
    print(f'{k}: {train_dataset[0][k].shape}')

print('train dataset length:', len(train_dataset))

Note that masking in the training dataset is stochastic, while masking in the test dataset is deterministic. This is implemented using a local_rng for the test dataset (see code)

The following training loss will be calculated only taking into account those sensors. This is implemented in the training script, not in the model nor the dataset.

Training the model

In [ ]:
import torch
from torch.utils.data import DataLoader
from stgae.model.bistgcn_opt import BiSTGCN
from stgae.model.train import train_step

# Hyperparameters
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR = "checkpoints" # Directory to store models

os.makedirs(SAVE_DIR, exist_ok=True)

# dataloader
dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True) #this does not shuffle timesteps inside samples, but samples inside the batches and improves SGD 

model = BiSTGCN(in_features=train_dataset.F, hidden_dim=32, adj=A).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
best_loss = float('inf')

print(f"Starting training on {DEVICE}...")

for epoch in range(EPOCHS):
    total_loss = 0.0
    
    for batch_idx, batch in enumerate(dataloader):
        loss = train_step(batch, model, optimizer, DEVICE)
        total_loss += loss
    avg_loss = total_loss / len(dataloader)

    if avg_loss < best_loss:
        best_loss = avg_loss
        best_path = os.path.join(SAVE_DIR, "bistgcn_best.pth")
        torch.save(model.state_dict(), best_path)
        print(f"   -> New best model saved to {best_path}")

    # 2. Save Checkpoint (For Resuming)
    # We save this every epoch. It includes optimizer state.
    checkpoint_path = os.path.join(SAVE_DIR, "bistgcn_last.pth")
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': avg_loss,
    }, checkpoint_path)

    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_loss:.6f}")

Evaluation

In [ ]:
# 1. Initialize the architecture (Must match training params!)
model = BiSTGCN(in_features=train_dataset.F, hidden_dim=32, adj=A).to(DEVICE)

model.load_state_dict(torch.load("checkpoints/bistgcn_best.pth"))

model.eval()

In [ ]:
#check example
print(filtered.iloc[0])

print(M[0,0]) #0 (sensor 0 not present at epoch 0)

print(X[1, 0, :])  # sensor 1, epoch 0, all features
print(M[1,0]) #1

In [ ]:
filtered[filtered['epoch']==0] #sensor 11 not present at epoch 0


NEXT STEP (CHECK GPT): Now implement thw windowed dataset